# Perturbation Cell and Tissue Atlas 综述数据探索

综述: https://doi.org/10.1016/j.cell.2024.07.035

本notebook展示综述引用的核心扰动数据分类、已有模块关系和数据查看方法。

In [ ]:
import pandas as pd
from pathlib import Path

base = Path('..')

# 加载数据清单
datasets = pd.read_csv(base / 'manifests' / 'datasets.csv')
print(f'=== 核心扰动数据 ({len(datasets)} 个) ===')
print(datasets[['dataset_id','name','perturbation_type','system','existing_module']].to_string(index=False))

## 数据分类统计

In [ ]:
print('按扰动类型:')
print(datasets['perturbation_type'].value_counts().to_string())

print('\n按模态:')
print(datasets['modality'].value_counts().to_string())

print('\n已有模块复用:')
has_module = datasets[datasets['existing_module'].notna() & (datasets['existing_module'] != '')]
print(f'  已有模块: {len(has_module)} 个')
print(f'  新增登记: {len(datasets) - len(has_module)} 个')
for _, row in has_module.iterrows():
    print(f'  {row["dataset_id"]} -> {row["existing_module"]}')

## 数据来源关系 (provenance)

In [ ]:
prov = pd.read_csv(base / 'manifests' / 'provenance.csv')
print(f'=== 数据来源关系 ({len(prov)} 条) ===')
print(prov[['source_id','relationship','target_id','target_module']].to_string(index=False))

print('\n=== 平台与工具 ===')
resources = pd.read_csv(base / 'manifests' / 'resources.csv')
print(resources[['name','type','purpose']].to_string(index=False))

## 查看一个已有数据集（Norman）

In [ ]:
# Norman 数据已在多个模块中登记
norman = datasets[datasets['dataset_id'] == 'norman_2019'].iloc[0]
print(f'Norman 2019:')
print(f'  编号: {norman["accession"]}')
print(f'  体系: {norman["system"]}')
print(f'  扰动: {norman["perturbation_type"]}')
print(f'  已有模块: {norman["existing_module"]}')
print(f'  入口: {norman["primary_url"]}')

# Norman 在 provenance 中的关系
norman_prov = prov[prov['source_id'] == 'norman_2019']
print(f'\nNorman 数据关系 ({len(norman_prov)} 条):')
for _, row in norman_prov.iterrows():
    print(f'  -> {row["target_id"]} ({row["relationship"]}, 模块: {row["target_module"]})')

## 待核实列表

In [ ]:
pending = pd.read_csv(base / 'manifests' / 'pending_verification.csv')
print(f'=== 待核实数据 ({len(pending)} 个) ===')
print(pending[['dataset_id','name','verification_priority']].to_string(index=False))
print('\n这些数据从综述参考文献定位原论文后，核对数据入口前标记为pending。')

## 数据处理原则提醒

In [ ]:
print('=== 数据处理原则 ===')
print('1. 保存原始与标准化标签的映射')
print('2. 不默认跨研究去批次')
print('3. 不把缺失标签补成对照')
print('4. 保留供者、样本、胚胎、实验批次及组合扰动信息')
print('5. 只有检查过文件内容后，才能标记原始计数或标准化表达')
print('6. 混合液滴不能默认当成单个细胞（Compressed Perturb-seq）')